# Curated Zinc-Finger Protein (ZFP) Analysis in *Caenorhabditis elegans*

## Scope

This notebook consolidates the curated ZFP workflow and the final PWM analysis of two tandem-repeat (TR) loci. It uses:

- **TR source:** the previously validated `TR_CATALOG_RAW.tsv` file only.
- **TF/PWM source:** the *C. elegans* CeMotifs archive, specifically `TF_Information_all_motifs_plus.txt` and the PWM matrices in `pwms_all_motifs/`.
- **PWM panel:** the 51 selected motif matrices listed explicitly below.

The notebook recovers selected TR records by chromosome coordinates, extracts complete monomers, ranks PWM matches on the consensus sequences of two TRs, and compares each complete monomer with the consensus of its assigned PWM.

**Interpretation:** a PWM sequence match indicates sequence compatibility. It does not, by itself, demonstrate in-vivo protein binding, regulatory activity, or a causal role in lineage specification.

## Execution

Run the notebook from top to bottom. In the first upload cell, provide either `TR_CATALOG_RAW.tsv` or a ZIP archive containing that exact file. The CeMotifs archive is downloaded in the next step.

## Consolidation notes

The exploratory non-ZFP screen, repeated 366-PWM short-fragment scans, per-TR duplicate code blocks, diagnostic variable dumps, and the final Z-DNA/PRE50 analyses have been removed. The curated 51-PWM selection and the final two-TR scoring workflow are retained in a compact, reproducible sequence.


In [ ]:
# 1. Load and validate the authoritative TR catalogue

import io
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import files

print("Upload TR_CATALOG_RAW.tsv or a ZIP archive containing it.")
uploaded_files = files.upload()

if len(uploaded_files) != 1:
    raise ValueError("Upload exactly one file.")

uploaded_name, uploaded_content = next(iter(uploaded_files.items()))

if uploaded_name.lower().endswith(".tsv"):
    normalized_name = (
        re.sub(r"\\s*\\(\\d+\\)$", "", uploaded_name.rsplit(".", 1)[0])
        + ".tsv"
    )
    if Path(normalized_name).name.upper() != "TR_CATALOG_RAW.TSV":
        raise ValueError(
            "The uploaded TSV must be TR_CATALOG_RAW.tsv "
            f"(received: {uploaded_name})."
        )
    raw = pd.read_csv(io.BytesIO(uploaded_content), sep="\\t")

elif uploaded_name.lower().endswith(".zip"):
    with zipfile.ZipFile(io.BytesIO(uploaded_content)) as archive:
        matches = [
            name for name in archive.namelist()
            if Path(name).name.upper() == "TR_CATALOG_RAW.TSV"
        ]
        if len(matches) != 1:
            raise ValueError(
                "The ZIP archive must contain exactly one "
                f"TR_CATALOG_RAW.tsv file; found {len(matches)}."
            )
        with archive.open(matches[0]) as source_file:
            raw = pd.read_csv(source_file, sep="\\t")
else:
    raise ValueError("Unsupported file type. Upload a TSV or ZIP archive.")

required_columns = {
    "Chromosome", "Start", "End", "Period",
    "Copy_Number", "Consensus", "Repeat_Sequence"
}
missing_columns = required_columns - set(raw.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

for column in ["Start", "End", "Period"]:
    raw[column] = pd.to_numeric(raw[column], errors="coerce")

print("TR_CATALOG_RAW validated")
print("Rows:", len(raw))
print("Columns:", len(raw.columns))
print("Uploaded file:", uploaded_name)


In [ ]:
# 2. Download CeMotifs and load the curated transcription-factor annotations

import requests

CEMOTIFS_URL = (
    "https://hugheslab.ccbr.utoronto.ca/"
    "supplementary-data/CeMotifs/CeMotifsData.zip"
)

response = requests.get(CEMOTIFS_URL, timeout=180)
response.raise_for_status()
CE_MOTIFS_ARCHIVE = response.content

with zipfile.ZipFile(io.BytesIO(CE_MOTIFS_ARCHIVE)) as archive:
    annotation_matches = [
        name for name in archive.namelist()
        if Path(name).name == "TF_Information_all_motifs_plus.txt"
    ]
    if len(annotation_matches) != 1:
        raise ValueError(
            "Expected one TF_Information_all_motifs_plus.txt file; "
            f"found {len(annotation_matches)}."
        )

    annotations = pd.read_csv(
        archive.open(annotation_matches[0]),
        sep="\\t",
        dtype=str,
        keep_default_na=False
    )

valid_pwm = annotations["Motif_ID"].str.fullmatch(r"M\\d+_1\\.02d")
annotations = annotations.loc[valid_pwm].copy()

print("CeMotifs archive downloaded:", len(CE_MOTIFS_ARCHIVE), "bytes")
print("Annotated PWM records:", len(annotations))
print("Distinct PWM IDs:", annotations["Motif_ID"].nunique())
print("Distinct annotated factors:", annotations["TF_Name"].nunique())


In [ ]:
# 3. Build the curated DNA-binding-domain catalogue and select the 51 PWM matrices

zinc_finger_pattern = (
    r"zf-C2H2|zf-C4|zf-BED|zf-C2HC|zf-CCCH|"
    r"zf-NF-X1|GATA|THAP|DM"
)

zfp_mask = annotations["DBDs"].str.contains(
    zinc_finger_pattern,
    case=False,
    regex=True,
    na=False
)
df_ZF = annotations.loc[zfp_mask].copy()

catalogue_columns = [
    "TF_Name", "Family_Name", "DBDs", "Motif_ID",
    "MSource_Identifier", "MSource_Type"
]
catalogo_ZF = (
    df_ZF[catalogue_columns]
    .drop_duplicates()
    .sort_values(["DBDs", "TF_Name", "Motif_ID"])
    .reset_index(drop=True)
)

family_summary = (
    df_ZF.groupby(["Family_Name", "DBDs"], dropna=False)
    .agg(
        Factors=("TF_Name", "nunique"),
        PWM_matrices=("Motif_ID", "nunique"),
        Annotation_records=("TF_Name", "size")
    )
    .reset_index()
    .sort_values(["Family_Name", "DBDs"])
)

print("Curated ZFP-related annotation records:", len(df_ZF))
print("Distinct factors:", df_ZF["TF_Name"].nunique())
print("Distinct PWM IDs:", df_ZF["Motif_ID"].nunique())
display(family_summary)

# Exact 51-matrix panel retained from the validated selection.
selected_pwm_numbers = (
    list(range(379, 390)) + [443] +
    list(range(445, 462)) +
    list(range(487, 500)) +
    list(range(544, 548)) +
    [2599, 4768, 4840, 4841, 6127]
)
ZNF_PWM_IDS = {
    f"M{number:04d}_1.02d" for number in selected_pwm_numbers
}
if len(ZNF_PWM_IDS) != 51:
    raise AssertionError(f"Expected 51 selected PWM IDs; found {len(ZNF_PWM_IDS)}.")

selected_catalogue = catalogo_ZF[
    catalogo_ZF["Motif_ID"].isin(ZNF_PWM_IDS)
].copy()

selected_pwm_found = set(selected_catalogue["Motif_ID"].dropna())
missing_selected_annotations = ZNF_PWM_IDS - selected_pwm_found
if missing_selected_annotations:
    raise ValueError(
        "Selected PWM IDs missing from the curated annotation catalogue: "
        f"{sorted(missing_selected_annotations)}"
    )

df_collegamento = (
    selected_catalogue.groupby("Motif_ID")
    .agg(
        Factors=("TF_Name", lambda values: "; ".join(sorted(set(values) - {""}))),
        Families=("Family_Name", lambda values: "; ".join(sorted(set(values) - {""}))),
        DBDs=("DBDs", lambda values: "; ".join(sorted(set(values) - {""})))
    )
    .reset_index()
    .rename(columns={"Motif_ID": "ID_PWM"})
)

# Load the physical PWM matrices from the same CeMotifs archive.
with zipfile.ZipFile(io.BytesIO(CE_MOTIFS_ARCHIVE)) as archive:
    pwm_file_map = {}
    for filename in archive.namelist():
        if (
            "pwms_all_motifs/" in filename
            and filename.endswith(".txt")
            and "__MACOSX" not in filename
            and "/._" not in filename
        ):
            match = re.search(r"(M\\d+_1\\.02d)", filename)
            if match:
                pwm_file_map[match.group(1)] = filename

    missing_pwm_files = ZNF_PWM_IDS - set(pwm_file_map)
    if missing_pwm_files:
        raise ValueError(
            "Selected PWM files missing from CeMotifs archive: "
            f"{sorted(missing_pwm_files)}"
        )

    pwms = {}
    for motif_id in sorted(ZNF_PWM_IDS):
        matrix_table = pd.read_csv(
            archive.open(pwm_file_map[motif_id]),
            sep="\\t"
        )
        matrix = (
            matrix_table[["A", "C", "G", "T"]]
            .apply(pd.to_numeric, errors="coerce")
            .to_numpy(dtype=float)
        )
        if matrix.ndim != 2 or matrix.shape[1] != 4 or not np.isfinite(matrix).all():
            raise ValueError(f"Invalid PWM matrix: {motif_id}")
        pwms[motif_id] = matrix

if len(pwms) != 51 or len(df_collegamento) != 51:
    raise AssertionError(
        f"Expected 51 PWM matrices and annotations; "
        f"loaded {len(pwms)} matrices and {len(df_collegamento)} annotations."
    )

print("Selected PWM matrices loaded:", len(pwms))
print("Selected PWM-to-factor associations:", len(df_collegamento))
display(df_collegamento.sort_values("ID_PWM").reset_index(drop=True))

# Explicitly check the developmental candidates discussed in the literature review.
developmental_candidates = {
    "DIE-1", "CES-1", "HAM-2", "EGL-43", "REF-2",
    "CHE-1", "PAG-3", "TRA-1", "ZTF-16", "LSL-1",
    "LIN-48", "LSY-2", "EOR-1"
}
candidate_rows = selected_catalogue[
    selected_catalogue["TF_Name"].str.upper().isin(developmental_candidates)
].copy()

found_candidates = set(candidate_rows["TF_Name"].str.upper())
print("Developmental candidates found in the 51-PWM panel:",
      sorted(found_candidates))
print("Candidates not represented in this panel:",
      sorted(developmental_candidates - found_candidates))

# MEX-6 is annotated in the broader catalogue, but its CCCH PWM is not in this panel.
assert "M0575_1.02d" not in ZNF_PWM_IDS


## Biological context: developmental factors within the selected ZFP set

The literature check identified several selected factors with developmental functions. The strongest matches to **embryonic events** are:

| Factor | DNA-binding family | Developmental evidence | Relevance to embryogenesis |
|---|---|---|---|
| **DIE-1** | C2H2 | Required for epithelial cell intercalation and embryonic elongation; mutant embryos show hypodermal, pharyngeal, gut, and muscle morphogenesis defects. | **Direct, strong** |
| **CES-1** | C2H2 / Snail | Controls asymmetric division of the embryonic NSM neuroblast and the size/fate of its daughters. | **Direct, strong** |
| **HAM-2** | Zinc-finger protein | Acts downstream of EGL-5 in HSN motor-neuron migration, an early differentiation event that includes embryonic development. | **Direct, strong** |
| **EGL-43** | Zinc-finger protein | Required for embryonic HSN migration and development of phasmid neurons; expressed in migrating HSNs. | **Direct, strong** |
| **REF-2** | ZIC-like zinc-finger protein | Required for normal cell-division timing and cell positioning in anterior embryonic lineages. | **Direct, strong** |
| **CHE-1** | C2H2 | Terminal selector for ASE sensory-neuron identity; its autoregulation helps initiate the ASE differentiation program during embryogenesis. | **Direct, strong** |
| **PAG-3** | C2H2 / GFI1-like | Regulates neuroblast fate, reiterative division patterns, and neuronal differentiation in the ventral nerve cord. | **Strong developmental-lineage relevance** |
| **TRA-1** | GLI-type zinc-finger protein | Present in embryonic somatic-gonad precursors Z1 and Z4; later regulates Z1 division polarity and contributes to SGP maturation. | **Embryonic expression; later lineage function** |
| **ZTF-16** | C2H2 / hunchback-Ikaros-like | Expressed in somatic-gonad precursors from embryogenesis; participates with EHN-3 in their development. | **Embryonic expression and gonadal development** |
| **LSL-1** | C2H2 | First detected in the P4 blastomere and present in primordial germ cells; regulates germline transcriptional programs. | **Direct embryonic germline expression; function extends later** |
| **LIN-48** | C2H2 | Required for hindgut and excretory-duct development; expression begins in relevant cells during late embryogenesis. | **Embryonic organogenesis** |
| **LSY-2** | C2H2 | Maintains the distinction between somatic and germline programs; loss of function allows somatic cells to acquire germline characteristics. | **Developmentally relevant; not a specific early-embryo phenotype** |
| **EOR-1** | PLZF-like zinc-finger protein | Regulates developmental signaling and neuronal fate programs. Some characterized functions concern later neuronal maturation or postembryonic development. | **Relevant, but less directly embryonic** |

### Additional candidate from the broader exploratory screen (outside the 51-PWM panel)

**EGRH-1** is a C2H2 zinc-finger transcription factor with a documented role in oocyte maturation and ovulation. Loss of function causes partially penetrant embryonic lethality, apparently in association with defects in reproductive physiology. Its PWM (`M3172_1.02d`) appeared in the earlier broad 366-PWM/AB_1745 short-fragment screen, with all 62 monomers covered, but **M3172_1.02d is not part of the selected 51-PWM panel**. It is therefore a relevant follow-up candidate, not a member of the final 51-matrix set.

### CCCH distinction

The full curated CeMotifs DNA-binding catalogue contains one annotated CCCH factor, **MEX-6** (`zf-CCCH`, PWM `M0575_1.02d`). However, `M0575_1.02d` is **not part of the selected 51-PWM panel**. It appeared in the earlier, broader AB_1745 PWM catalogue screen, not in this exact 51-matrix selection.

Other embryonic CCCH proteins include MEX-5, MEX-1, POS-1, PIE-1, OMA-1, and OMA-2. These proteins act mainly as RNA-binding or translational regulators of maternal transcripts and cell-fate determinants; they are not PWM-selected DNA-binding transcription factors in this notebook. MEX-5 and MEX-6 are especially relevant to anterior-posterior polarity and the asymmetric distribution of embryonic determinants.

### References

- DIE-1 and embryonic epithelial rearrangement: [Heid et al., 2001, *Developmental Biology*](https://pubmed.ncbi.nlm.nih.gov/11456452/).\n- EGL-43 and embryonic HSN migration: [Garriga et al., 1993, *Genes & Development*](https://pubmed.ncbi.nlm.nih.gov/8224840/).\n- REF-2 and anterior embryonic lineages: [POP-1/TCF, REF-2/ZIC and T-box factors](https://pmc.ncbi.nlm.nih.gov/articles/PMC9378603/).\n- CHE-1 and embryonic ASE-neuron differentiation: [Transcription factor autoregulation and neuronal identity](https://pmc.ncbi.nlm.nih.gov/articles/PMC6633604/).\n- PAG-3 and neuroblast fate: [PAG-3, a Zn-finger transcription factor](https://pubmed.ncbi.nlm.nih.gov/11923211/).\n- LSL-1 and embryonic germline transcription: [Rodriguez Crespo et al., 2022, *Genetics*](https://pmc.ncbi.nlm.nih.gov/articles/PMC9071529/).\n- LIN-48 and hindgut/excretory organ development: [EGL-38 Pax and LIN-48](https://pubmed.ncbi.nlm.nih.gov/11532910/).
- CES-1 and asymmetric embryonic NSM neuroblast division: [Wei et al., 2017, *Genetics*](https://academic.oup.com/genetics/article/206/4/2069/6072598).
- HAM-2 and HSN migration: [The C. elegans gene ham-2 links Hox patterning to migration of the HSN motor neuron](https://pubmed.ncbi.nlm.nih.gov/10049362/).
- TRA-1, Z1/Z4 polarity, and somatic-gonad precursor development: [Mechanisms of lineage specification in *C. elegans*](https://academic.oup.com/genetics/article/225/4/iyad174/7320268).
- ZTF-16 and somatic-gonad development: [Hunchback and Ikaros-like zinc finger genes control reproductive system development](https://pmc.ncbi.nlm.nih.gov/articles/PMC3721651/).
- LSY-2 and germ-soma distinction: [Lin et al., 2015, *Protein & Cell*](https://pubmed.ncbi.nlm.nih.gov/26050091/).
- EOR-1 and developmental/neuronal fate regulation: [EOR-1/PLZF and neuronal maturation](https://pmc.ncbi.nlm.nih.gov/articles/PMC5962631/).\n- EGRH-1, oocyte maturation/ovulation, and partially penetrant embryonic lethality: [The EGR family gene egrh-1](https://pmc.ncbi.nlm.nih.gov/articles/PMC2926961/).
- CCCH MEX-5/MEX-6 and embryonic polarity: [Reduced dosage of pos-1 suppresses Mex mutants](https://pubmed.ncbi.nlm.nih.gov/17028349/); [MEX-5/6 in early embryonic polarity](https://www.ncbi.nlm.nih.gov/books/NBK19708/).\n- CCCH POS-1 and embryonic maternal-RNA regulation: [RNA target specificity of POS-1](https://pubmed.ncbi.nlm.nih.gov/18952820/).

These are literature-based biological annotations, not evidence that any factor binds the TR loci examined here.


## TR recovery and monomer extraction

The following register consolidates the 15 physical loci previously validated in the notebook and adds the additional chromosome-X locus examined in the final extension. Each locus is recovered from `TR_CATALOG_RAW.tsv` by chromosome, start, end, and expected period. No sequence is reconstructed from coordinates or external sources.

Only complete monomers are retained. Any incomplete terminal tail is recorded and excluded from monomer-level scoring.


In [ ]:
# 4. Recover the selected TR loci directly from TR_CATALOG_RAW

selected_tr_records = [
    # TR ID, chromosome, start, end, expected period, lineage/role
    ("AB_1745", "I", 1919838, 1922036, 35, "AB — Pre-APO"),
    ("AB_32946", "V", 18567606, 18568227, 10, "AB — Pre-APO"),
    ("AB_33142", "V", 18694160, 18694786, 10, "AB — Pre-APO"),
    ("MS_33627", "V", 19122509, 19124517, 66, "MS — Pre-APO"),
    ("C_10956", "II", 12197271, 12197445, 12, "C — Pre-APO"),
    ("D_32101", "V", 17361885, 17361931, 7, "D — Pre-APO"),
    ("E_10203", "II", 8833974, 8834023, 8, "E — Pre-APO"),
    ("TR_IV_6682656_6731929", "IV", 6682656, 6731929, 174, "AB — STOP/DIVIDE"),
    ("TR_III_13584282_13599363", "III", 13584282, 13599363, 51, "AB — STOP/DIVIDE"),
    ("TR_III_13584328_13599377", "III", 13584328, 13599377, 16, "AB — STOP/DIVIDE"),
    ("TR_IV_1902960_1903281", "IV", 1902960, 1903281, 6, "AB — STOP/DIVIDE"),
    ("TR_III_1239493_1240806", "III", 1239493, 1240806, 35, "MS — STOP/DIVIDE; MS — APO"),
    ("TR_I_355869_356717", "I", 355869, 356717, 19, "C — STOP/DIVIDE; C — APO"),
    ("TR_IV_6682656_6704725", "IV", 6682656, 6704725, 172, "D — STOP/DIVIDE; E — STOP/DIVIDE"),
    ("TR_X_4031472_4056451", "X", 4031472, 4056451, 20, "AB — APO"),
    ("TR_X_4668814_4670218", "X", 4668814, 4670218, 17, "Additional TR"),
]

tr_rows = []

for tr_id, chromosome, start, end, expected_period, roles in selected_tr_records:
    chromosome_values = (
        raw["Chromosome"].astype(str)
        .str.replace(r"^chr", "", regex=True, case=False)
    )
    hits = raw.loc[
        (chromosome_values == chromosome)
        & (raw["Start"] == start)
        & (raw["End"] == end)
        & (raw["Period"] == expected_period)
    ]

    if len(hits) != 1:
        raise ValueError(
            f"{tr_id}: expected exactly one RAW record "
            f"for {chromosome}:{start}-{end}, period {expected_period}; "
            f"found {len(hits)}."
        )

    row = hits.iloc[0]
    repeat_sequence = str(row["Repeat_Sequence"]).upper()
    consensus = str(row["Consensus"]).upper()
    period = int(row["Period"])

    if not repeat_sequence or period <= 0:
        raise ValueError(f"{tr_id}: empty sequence or invalid period.")
    if set(repeat_sequence) - set("ACGT"):
        raise ValueError(f"{tr_id}: non-ACGT symbols in Repeat_Sequence.")

    complete_monomers = len(repeat_sequence) // period
    complete_length = complete_monomers * period
    terminal_tail = len(repeat_sequence) - complete_length

    tr_rows.append({
        "TR_ID": tr_id,
        "Roles": roles,
        "Chromosome": chromosome,
        "Start": start,
        "End": end,
        "Period": period,
        "Copy_Number": row["Copy_Number"],
        "Consensus": consensus,
        "Repeat_Sequence": repeat_sequence,
        "Original_length_nt": len(repeat_sequence),
        "Complete_monomers": complete_monomers,
        "Complete_sequence_length_nt": complete_length,
        "Terminal_tail_nt": terminal_tail
    })

df_selected_TRs = pd.DataFrame(tr_rows)

print("Physical TR loci recovered:", len(df_selected_TRs))
print("Complete monomers:", int(df_selected_TRs["Complete_monomers"].sum()))
print("Terminal tail nucleotides excluded:",
      int(df_selected_TRs["Terminal_tail_nt"].sum()))
display(
    df_selected_TRs[
        ["TR_ID", "Roles", "Chromosome", "Start", "End",
         "Period", "Complete_monomers", "Terminal_tail_nt"]
    ]
)


In [ ]:
# 5. Extract and structurally validate every complete monomer

monomer_rows = []

for _, tr in df_selected_TRs.iterrows():
    sequence = tr["Repeat_Sequence"]
    period = int(tr["Period"])
    number_of_monomers = int(tr["Complete_monomers"])

    for index in range(number_of_monomers):
        start0 = index * period
        end0 = start0 + period
        monomer_sequence = sequence[start0:end0]

        if len(monomer_sequence) != period:
            raise ValueError(
                f"{tr['TR_ID']}: monomer {index + 1} has an invalid length."
            )

        monomer_rows.append({
            "TR_ID": tr["TR_ID"],
            "Roles": tr["Roles"],
            "Chromosome": tr["Chromosome"],
            "TR_Start": int(tr["Start"]),
            "TR_End": int(tr["End"]),
            "Period": period,
            "Monomer": index + 1,
            "Monomer_start_1based": start0 + 1,
            "Monomer_end_1based": end0,
            "Sequence": monomer_sequence,
            "Length": len(monomer_sequence)
        })

MONOMERS = pd.DataFrame(monomer_rows)

expected_counts = df_selected_TRs.set_index("TR_ID")["Complete_monomers"]
observed_counts = MONOMERS.groupby("TR_ID")["Monomer"].nunique()

if not observed_counts.reindex(expected_counts.index).equals(
    expected_counts.astype(int)
):
    raise ValueError("Extracted monomer counts do not match the expected counts.")

if not (MONOMERS["Length"] == MONOMERS["Period"]).all():
    raise ValueError("At least one extracted monomer has an invalid length.")

# Reconstruct each complete sequence from the extracted monomers.
for _, tr in df_selected_TRs.iterrows():
    extracted = "".join(
        MONOMERS.loc[
            MONOMERS["TR_ID"] == tr["TR_ID"], "Sequence"
        ].tolist()
    )
    expected = tr["Repeat_Sequence"][:int(tr["Complete_sequence_length_nt"])]
    if extracted != expected:
        raise ValueError(f"{tr['TR_ID']}: monomer reconstruction check failed.")

monomer_summary = (
    MONOMERS.groupby("TR_ID")
    .agg(
        Roles=("Roles", "first"),
        Monomers=("Monomer", "nunique"),
        Period=("Period", "first"),
        Nucleotides=("Length", "sum")
    )
    .reset_index()
)

print("TR loci validated:", len(monomer_summary))
print("Monomers extracted:", len(MONOMERS))
print("Nucleotides in complete monomers:", int(MONOMERS["Length"].sum()))
display(monomer_summary)


## PWM scoring of the two TR consensus sequences

The final extension in the source notebook examined:

- `TR_IV_6682656_6704725` (128 complete monomers; period 172 nt)
- `TR_X_4668814_4670218` (82 complete monomers; period 17 nt)

For each selected PWM, the best-scoring window is identified on both strands of the TR consensus. The relative score is calculated from the PWM's position-wise minimum and maximum log-odds scores. The selected PWM assignments are then compared with every complete monomer.

The assignments retained from the validated run are `M0493_1.02d` (LIN-29) for the chromosome-IV TR and `M0380_1.02d` (HAM-2 / ZTF-30) for the chromosome-X TR. These are candidate sequence associations, not confirmed binding events.


In [ ]:
# 6. Rank all 51 PWM matrices against the two TR consensus sequences

BASES = "ACGT"
BACKGROUND_PROBABILITY = 0.25
COMPLEMENT = str.maketrans("ACGT", "TGCA")

def reverse_complement(sequence):
    return sequence.translate(COMPLEMENT)[::-1]

def pwm_log_odds(matrix):
    matrix = np.asarray(matrix, dtype=float)
    return np.log2(np.maximum(matrix, 1e-12) / BACKGROUND_PROBABILITY)

target_TR_ids = [
    "TR_IV_6682656_6704725",
    "TR_X_4668814_4670218"
]

scan_rows = []

for tr_id in target_TR_ids:
    tr_row = df_selected_TRs.loc[
        df_selected_TRs["TR_ID"] == tr_id
    ]
    if len(tr_row) != 1:
        raise ValueError(f"Expected one record for {tr_id}.")

    consensus_sequence = str(tr_row.iloc[0]["Consensus"])

    for pwm_id, matrix in pwms.items():
        matrix = np.asarray(matrix, dtype=float)
        motif_length = len(matrix)

        if motif_length > len(consensus_sequence):
            continue

        log_odds = pwm_log_odds(matrix)
        minimum_score = log_odds.min(axis=1).sum()
        maximum_score = log_odds.max(axis=1).sum()

        best_candidate = None

        for strand, scanned_sequence in [
            ("+", consensus_sequence),
            ("-", reverse_complement(consensus_sequence))
        ]:
            for start in range(len(scanned_sequence) - motif_length + 1):
                window = scanned_sequence[start:start + motif_length]
                if set(window) - set(BASES):
                    continue

                score = sum(
                    log_odds[position, BASES.index(base)]
                    for position, base in enumerate(window)
                )

                relative_score = (
                    100 * (score - minimum_score) /
                    (maximum_score - minimum_score)
                    if maximum_score > minimum_score else np.nan
                )

                candidate = (
                    score, relative_score, strand, start + 1, window
                )
                if best_candidate is None or candidate[0] > best_candidate[0]:
                    best_candidate = candidate

        if best_candidate is not None:
            score, relative_score, strand, position, matched_sequence = best_candidate
            scan_rows.append({
                "TR_ID": tr_id,
                "PWM": pwm_id,
                "Score": score,
                "Relative_score_pct": relative_score,
                "Strand": strand,
                "Consensus_position": position,
                "Matched_sequence": matched_sequence
            })

df_consensus_scan = pd.DataFrame(scan_rows).merge(
    df_collegamento,
    left_on="PWM",
    right_on="ID_PWM",
    how="left"
).drop(columns="ID_PWM")

df_consensus_scan = df_consensus_scan.sort_values(
    ["TR_ID", "Relative_score_pct"],
    ascending=[True, False]
).reset_index(drop=True)

for tr_id in target_TR_ids:
    print("\\nTop PWM candidates:", tr_id)
    display(
        df_consensus_scan.loc[
            df_consensus_scan["TR_ID"] == tr_id
        ].head(10)
    )


In [ ]:
# 7. Score each complete monomer with its assigned PWM

# Assignments retained from the validated source notebook.
PWM_ASSIGNMENTS = {
    "TR_IV_6682656_6704725": "M0493_1.02d",
    "TR_X_4668814_4670218": "M0380_1.02d"
}

# Confirm that the chromosome-X assignment remains the top-ranked candidate.
top_X_pwm = df_consensus_scan.loc[
    df_consensus_scan["TR_ID"] == "TR_X_4668814_4670218",
    "PWM"
].iloc[0]
if top_X_pwm != PWM_ASSIGNMENTS["TR_X_4668814_4670218"]:
    raise ValueError(
        "The top-ranked chromosome-X PWM differs from the retained assignment: "
        f"{top_X_pwm}."
    )

monomer_score_rows = []

for tr_id, pwm_id in PWM_ASSIGNMENTS.items():
    if pwm_id not in pwms:
        raise ValueError(f"Assigned PWM is not loaded: {pwm_id}")

    matrix = np.asarray(pwms[pwm_id], dtype=float)
    motif_length = len(matrix)
    log_odds = pwm_log_odds(matrix)

    pwm_consensus = "".join(
        BASES[index] for index in np.argmax(matrix, axis=1)
    )

    tr_monomers = (
        MONOMERS.loc[MONOMERS["TR_ID"] == tr_id]
        .sort_values("Monomer")
    )

    for _, monomer in tr_monomers.iterrows():
        sequence = monomer["Sequence"]
        if motif_length > len(sequence):
            raise ValueError(
                f"{tr_id}: PWM length exceeds monomer length."
            )

        candidates = []

        for strand, scanned_sequence in [
            ("+", sequence),
            ("-", reverse_complement(sequence))
        ]:
            for start in range(len(scanned_sequence) - motif_length + 1):
                window = scanned_sequence[start:start + motif_length]
                if set(window) - set(BASES):
                    continue

                score = sum(
                    log_odds[position, BASES.index(base)]
                    for position, base in enumerate(window)
                )
                candidates.append(
                    (score, strand, start + 1, window)
                )

        if not candidates:
            continue

        score, strand, position, matched_sequence = max(
            candidates, key=lambda item: item[0]
        )
        differences = sum(
            observed != expected
            for observed, expected in zip(matched_sequence, pwm_consensus)
        )

        monomer_score_rows.append({
            "TR_ID": tr_id,
            "PWM": pwm_id,
            "Factor": df_collegamento.loc[
                df_collegamento["ID_PWM"] == pwm_id, "Factors"
            ].iloc[0],
            "Monomer": int(monomer["Monomer"]),
            "Strand": strand,
            "Position": position,
            "Matched_sequence": matched_sequence,
            "PWM_consensus": pwm_consensus,
            "Differences": differences
        })

df_monomer_scores = pd.DataFrame(monomer_score_rows)

summary_rows = []
for tr_id, group in df_monomer_scores.groupby("TR_ID", sort=False):
    summary_rows.append({
        "TR_ID": tr_id,
        "PWM": PWM_ASSIGNMENTS[tr_id],
        "Factor": group["Factor"].iloc[0],
        "Monomers_scored": group["Monomer"].nunique(),
        "Within_3_nt": int((group["Differences"] <= 3).sum()),
        "Within_4_nt": int((group["Differences"] <= 4).sum()),
        "Exact_consensus_matches": int((group["Differences"] == 0).sum())
    })

df_monomer_summary = pd.DataFrame(summary_rows)

print("Final monomer-level summary")
display(df_monomer_summary)

print("Monomer-level scoring details")
display(df_monomer_scores)


In [ ]:
# 8. Export the final tables

df_selected_TRs.drop(columns=["Repeat_Sequence"]).to_csv(
    "selected_TR_catalogue.csv", index=False
)
monomer_summary.to_csv(
    "selected_TR_monomer_summary.csv", index=False
)
df_consensus_scan.to_csv(
    "selected_51_PWM_consensus_scan.csv", index=False
)
df_monomer_summary.to_csv(
    "selected_TR_ZFP_summary.csv", index=False
)
df_monomer_scores.to_csv(
    "selected_TR_ZFP_monomer_scores.csv", index=False
)

print("Exported:")
for filename in [
    "selected_TR_catalogue.csv",
    "selected_TR_monomer_summary.csv",
    "selected_51_PWM_consensus_scan.csv",
    "selected_TR_ZFP_summary.csv",
    "selected_TR_ZFP_monomer_scores.csv"
]:
    print("-", filename)
